<a href="https://colab.research.google.com/github/john-cant/AMLS2_24_25_SNJCCAN57/blob/main/Track2_Tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# AMLS2 Assignment
## Track 2 Script

Runs Track 2 SRReNet Tune model.

## Import libraries
The required libraries for this notebook are sklearn, copy, numpy and matplotlib.

In [7]:
## first enable autoreload during development so latest (new) version local code library is reloaded on execution
## can be commented out when local code development not happening to avoid overhead
%load_ext autoreload
%autoreload 2

from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

## import libraries
import io
import glob
import os
import numpy as np
import matplotlib.pyplot as plt

from google.colab import drive
if not os.path.exists('/content/drive/My Drive'):   ## check if Google drive mounted
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Google Drive is already mounted.")
import sys

sys.path.append('/content/drive/My Drive/AMLS2')       ## load project directory
## load additional functions I have developed to support AMLS assignments
import AMLS_common as ac

## import tensorflow
import tensorflow as tf
print(tf.__version__)
from tensorflow.keras import models
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Multiply, Add, Layer, Lambda, LeakyReLU
from tensorflow.keras.layers import Input, Conv2D, Flatten, UpSampling2D, Dropout, BatchNormalization, PReLU
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.losses import BinaryCrossentropy, Hinge, MeanAbsoluteError
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.initializers import Constant
from tensorflow.keras.applications.vgg19 import VGG19
import tensorflow.keras.backend as K
from tensorflow.nn import depth_to_space

from skimage.metrics import structural_similarity as ssim


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Your runtime has 89.6 gigabytes of available RAM

You are using a high-RAM runtime!
Thu Apr  3 12:45:37 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P0             50W /  400W |   39039MiB /  40

## Set base parameters
Including hyperparameters and environment specifics

In [8]:
folderbase   = '/content/drive/My Drive/AMLS2/'            ## project folder
## Initialize hyperparameters from spreadsheet
filename     = folderbase+"hyperparam_base_2.xlsx"
if os.path.isfile(filename):
    item = ac.HyperParameters.load_excel(filename)
else:
    print(filename+" does not exist. Loading default hyperparameters instead")
    item      = ac.HyperParameters(learning_rate=0.01,
                                   kernel_size=3,
                                   num_epochs=75,
                                   num_filter=128,
                                   layers=4,
                                   dropout_rate=0.25,
                                   optimise="Adam",
                                   strides=2,
                                   padding="same",
                                   loss="Adam")
## print out hyperparameters
print(ac.HyperParameters.list_parameters(item))
## set up progress tracking call back
tqdm_callback = ac.TqdmEpochProgress(total_epochs=item.num_epochs)

learning_rate: 0.001
batch_size: 2
num_epochs: 50
optimise: Adam
loss: ssim_loss_plus
num_filter: 128
strides: 1
padding: same
dropout_rate: 0.25
layers: 64
activation: prelu
kernel_size: 3
scale: 4
momentum: 0.9
epsilon: 1e-05



In [9]:
## control (e.g. verbose) parameters
filebase   = "/content/drive/My Drive/AMLS2/metrics/"          ## place to save output files
verbose    = 1                   ## to control whether additional in process information is printed

## Load and preprocess the Data
We load the dataset.

In [10]:
## Loading the data file using a loader
UPSCALE_FACTOR = 2           ## Factor between LR and HR
BATCH_SIZE     = 4
IMG_SIZE       = 224
CROP_SIZE      = 224         ## HR crop size
# Define training folder paths
lr_train_folder = "/content/drive/My Drive/AMLS2/dataset/track2/train/LR/DIV2K_train_LR_unknown/X2"
hr_train_folder = "/content/drive/My Drive/AMLS2/dataset/track1/train/HR/DIV2K_train_HR"

# Define validation folder paths
lr_val_folder = "/content/drive/My Drive/AMLS2/dataset/track2/val/LR/DIV2K_valid_LR_unknown/X2"
hr_val_folder = "/content/drive/My Drive/AMLS2/dataset/track1/val/HR/DIV2K_valid_HR"

train_dataset,val_dataset = ac.load_data(lr_train_folder,hr_train_folder,lr_val_folder,hr_val_folder,BATCH_SIZE,UPSCALE_FACTOR)
verbose = 1
if verbose == 1:
    ## print summary stats for training dataset
    print("\nSummary metrics for train_dataset")
    print("type:",type(train_dataset))
    print("length:",len(train_dataset))
    print("shape:",train_dataset)

print("end load")
### end tested load


Summary metrics for train_dataset
type: <class 'tensorflow.python.data.ops.prefetch_op._PrefetchDataset'>
length: 200
shape: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 448, 448, 3), dtype=tf.float32, name=None))>

Summary metrics for train_dataset
type: <class 'tensorflow.python.data.ops.prefetch_op._PrefetchDataset'>
length: 200
shape: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 448, 448, 3), dtype=tf.float32, name=None))>
end load


## Define the model

Define a base model

In [11]:
## Define the model
choice = "1"

if choice == "1":
  ##batch_size = 4
  model = ac.srresnet_tune_2(item)
  # Add resizing layer to match output to target dimensions
  out   = tf.keras.layers.Resizing(448,448)(model.output)
  model = tf.keras.models.Model(inputs=model.input, outputs=out)
  model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=item.learning_rate), # Use item.learning_rate
                loss= ac.ssim_loss_plus,  # ssim loss
                metrics=['acc'])
if choice == "2":
  model = ac.edsr_plus(item)
  model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=item.learning_rate), # Use item.learning_rate
                loss= ac.ssim_loss,  # ssim loss
                metrics=['acc'])

for lowres, highres in train_dataset.take(1):  # Take one batch
    lowres_input = lowres[0]  # Get the first image
    preds = model(tf.expand_dims(lowres_input, axis=0))  # Predict on it
    print("** Model output shape:", preds.shape)
    print("** Target shape:", highres[0].shape)  # Get shape of target image
    break
## Redirect the summary output to a string to be saved to file
summary_string = io.StringIO()
model.summary(print_fn=lambda x: summary_string.write(x + "\n"))
summary_content = summary_string.getvalue()
summary_string.close()

activation prelu
** Model output shape: (1, 448, 448, 3)
** Target shape: (448, 448, 3)


## Fit the model

Fit using hyperparameters as defined above

In [ ]:
## Fit the model
steps_per_epoch  = len(train_dataset)  ## Understand dataset already adjusted by batch
validation_steps = len(val_dataset)    ## Also set up validation

if verbose == 1:
    print(item.num_epochs)
    print("steps",steps_per_epoch,"val_steps",validation_steps)

history = model.fit(
    train_dataset,
    epochs=item.num_epochs,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_dataset,
    validation_steps=validation_steps,
    verbose=1,                                  ## Set to 1 for progress updates
    callbacks=[tqdm_callback]
)


In [ ]:
## output graphs and save metrics files
ac.graph_and_save(history,summary_content,item,filebase)

In [ ]:
ac.test_model(val_dataset,model,item.batch_size,filebase)